In [1]:
import pandas as pd
import os
from pathlib import Path
import chardet
import re
print("current directory listing, locate your cve data file:")
print(f"=" * 70)
current_dir = Path('.')
for item in current_dir.iterdir():
    print(item.name)


current directory listing, locate your cve data file:
.ipynb_checkpoints
2025-ratings-sep-14.txt
causality.ipynb
causality_output_2026-03-17.csv
causality_output_2026-03-19.csv
causality_output_2026-03-20.csv
counter.ipynb
lines_with_wrong_comma_count.txt
pdates.md
reduction.txt
times.csv
times_new.csv
times_scratch.md
vdata.csv
vuln-analysis.ipynb


### CAUSALITY Prediction Counter
Specify your CVE data in the next field, identify the field  containing CVEs which the fourth cell will try to identify for you.  The final output will be a count of predictions that exist in the local population. 

In [2]:
vuln_path = 'vdata.csv' # Identify the vuln data file to ingest

In [3]:
with open(vuln_path, 'rb') as file:
    result = chardet.detect(file.read())
    encoding = result['encoding']
vulns = pd.read_csv(vuln_path, low_memory=False, encoding=encoding, header=0)
vulns.columns = vulns.columns.str.strip().str.lower()

all_nan = [col for col in vulns.columns if vulns[col].isna().all()]
if all_nan:
    print("\n⚠️ Fields entirely NaN:")
    for col in all_nan:
        print(f" - {col}")
else:
    print("\n✅ No fields are entirely NaN")

print("Shape of the vulns dataframe is:", vulns.shape)
print()
print("These are the fields in your dataframe:")
print(vulns.columns.tolist())


✅ No fields are entirely NaN
Shape of the vulns dataframe is: (3899, 6)

These are the fields in your dataframe:
['cve', 'vendorproject', 'product', 'shortdescription', 'rating', 'vulnerabilityname']


In [4]:
# CVE pattern: CVE-YYYY-#####
cve_pattern = r'CVE-\d{4}-\d{4,7}'

# Search for the column containing CVEs
cve_column = None

for col in vulns.columns:
    try:
        # Convert to string and check for CVE pattern
        sample = vulns[col].dropna().head(50).astype(str)
        if sample.str.contains(cve_pattern, regex=True, na=False).any():
            cve_column = col
            break
    except:
        pass

if cve_column:
    print(f"✓ CVE field found: '{cve_column}'")
    print(f"\nFirst 5 values:")
    for i, cve in enumerate(vulns[cve_column].head(5), 1):
        print(f"  {i}. {cve}")
else:
    print("❌ No column with CVE pattern found")
    print("\nAvailable columns:")
    print(vulns.columns.tolist())
    print("\nSample data:")
    print(vulns.head())

✓ CVE field found: 'cve'

First 5 values:
  1. CVE-2024-4535
  2. CVE-2024-11821
  3. CVE-2024-2531
  4. CVE-2024-9873
  5. CVE-2024-9073


In the output above, identify the your field name that contains CVEs and specify it in the cell below.

In [5]:
# Check out the field list above and identify your field that contains CVE IDs. Provide it to the function below
# so that we have normalized field names across dataframes.

SOURCE_CVE_FIELD = 'cve'  # <-- change this as needed
colmap = {c.lower(): c for c in vulns.columns}

if SOURCE_CVE_FIELD.lower() in colmap:
    src = colmap[SOURCE_CVE_FIELD.lower()]
    if src == 'cve':
        pass  # already named 'cve'
    elif 'cve' in vulns.columns:
        print("Target column 'cve' already exists; skipping rename to avoid duplicate.")
    else:
        vulns.rename(columns={src: 'cve'}, inplace=True)
else:
    print(f"Column '{SOURCE_CVE_FIELD}' not found; nothing to rename.")

In [6]:
predictions = pd.read_csv('times_new.csv')
predictions.columns = predictions.columns.str.lower()
predictions['leadtimedays'] = predictions['leadtimedays'].astype(int)
print(f"Predictions loaded: {len(predictions)} rows")
print(f"Predictions columns: {predictions.columns.tolist()}")

Predictions loaded: 206 rows
Predictions columns: ['cve', 'leadtimedays', 'pred_date', 'kev_date']


In [7]:
vulns_cves = vulns['cve'].unique()
# Filter predictions to only rows with CVEs that are in vulns
matching_predictions = predictions[predictions['cve'].isin(vulns_cves)]

print(f"This is the number we are interested in, you have: {len(matching_predictions)} prediction hits!")
print()
# Calculate statistics
leadtime = matching_predictions['leadtimedays'].dropna()
print(f"Lead Time Statistics (Matching Predictions)")
print(f"These numbers show how much early warning you would have had:")
print(f"=" * 50)
print(f"Count: {len(leadtime)}")
print(f"Mean: {leadtime.mean():.2f} days")
print(f"Median: {leadtime.median():.2f} days")
print(f"Std Dev: {leadtime.std():.2f} days")
print(f"Min: {leadtime.min():.0f} days")
print(f"Max: {leadtime.max():.0f} days")
print()
matching_predictions

This is the number we are interested in, you have: 17 prediction hits!

Lead Time Statistics (Matching Predictions)
These numbers show how much early warning you would have had:
Count: 17
Mean: 86.29 days
Median: 56.00 days
Std Dev: 83.75 days
Min: 5 days
Max: 263 days



,cve,leadtimedays,pred_date,kev_date
0,CVE-2024-0582,54,01/03/2025,02/26/2025
49,CVE-2024-34193,185,01/17/2025,07/21/2025
50,CVE-2024-34257,223,01/07/2025,08/18/2025
55,CVE-2024-38100,54,01/03/2025,02/26/2025
78,CVE-2024-50302,56,01/07/2025,03/04/2025
83,CVE-2024-53150,92,01/07/2025,04/09/2025
87,CVE-2024-53704,32,01/17/2025,02/18/2025
94,CVE-2024-56159,263,05/24/2025,02/11/2026
95,CVE-2024-5827,228,01/03/2025,08/19/2025
96,CVE-2024-5932,71,01/07/2025,03/19/2025
